# This notebook computes covariance matrices for each of the BM tree nodes


In [ ]:
import os
import copy
import re
import pickle
import numpy as np
import pandas as pd
import shutil
import glob
import subprocess

import rmgpy.data.thermo
import rmgpy.data.rmg
import rmgpy.chemkin

import rmgpy.tools.uncertainty

import matplotlib.pyplot as plt
%matplotlib inline

# 1. Load the database

In [ ]:
# load the tree from the database
database = rmgpy.data.rmg.RMGDatabase()
database.load(
    path = rmgpy.settings['database.directory'],
    thermo_libraries = [
        'Klippenstein_Glarborg2016',
        'BurkeH2O2',
        'thermo_DFT_CCSDTF12_BAC', 
        'DFT_QCI_thermo',
        'primaryThermoLibrary',
        'primaryNS',
        'NitrogenCurran',
        'NOx2018',
        'FFCM1(-)',
        'SulfurLibrary',
        'SulfurGlarborgH2S',
        'SABIC_aromatics'
    ],
    transport_libraries = [],
    reaction_libraries = [],
    seed_mechanisms = [],
    kinetics_families = 'default',
    kinetics_depositories = ['training'],
    depository = False,
) 

for family in database.kinetics.families:
    if not database.kinetics.families[family].auto_generated:
        database.kinetics.families[family].add_rules_from_training(thermo_database=database.thermo)
        database.kinetics.families[family].fill_rules_by_averaging_up(verbose=True)

# 2. Get the mapping of how training reactions fall on the Tree

In [ ]:
for f in database.kinetics.families:
    if database.kinetics.families[f].auto_generated:
        print(f)

In [ ]:
family_name = '2+2_cycloaddition'  # 72.8 % are not reproduced...
# family_name = 'CO_Disproportionation'


reaction_maps = database.kinetics.families[family_name].get_reaction_matches(
    thermo_database=database.thermo,
    remove_degeneracy=True,
    get_reverse=True,
    exact_matches_only=False,
    fix_labels=True
)
training_reaction_items = [x.item for key, x in database.kinetics.families[family_name].depositories[0].entries.items()]
training_reaction_data = [x.data for key, x in database.kinetics.families[family_name].depositories[0].entries.items()]

In [ ]:
[x.index for x in training_reaction_items]

In [ ]:
training_reaction_items[0].products[0].molecule[0].get_all_labeled_atoms()

In [ ]:
training_reaction_items[2].products[0].molecule[0].get_all_labeled_atoms()

In [ ]:
training_reaction_items[0].is_isomorphic(training_reaction_items[2], check_identical=True)

In [ ]:
training_reaction_items[0]

In [ ]:
reaction_maps.keys()

In [ ]:
for i in range(len(reaction_maps['Root'])):
    print(hex(id(reaction_maps['Root'][i])))

In [ ]:
for i in range(len(reaction_maps['Root_1COCSCdCdd->Cd'])):
    print(hex(id(reaction_maps['Root_1COCSCdCdd->Cd'][i])))

In [ ]:
ensemble.shape

In [ ]:
ensemble_files = sorted(glob.glob('/home/moon/uncertainty_estimator/BM_covariances/*_ensemble_matrix.npy'))
for i in range(len(ensemble_files)):

    ensemble = np.load(ensemble_files[i])


    log_ensemble = np.log(ensemble)
    log_cov = np.cov(log_ensemble)
    plt.matshow(log_cov)
    plt.colorbar()
    plt.clim([-0.25, .25])
    plt.title(ensemble_files[i])
    # break
    plt.show()
    
# cov = np.load('Disproportionation_covariance_matrix.npy')

# with open('Disproportionation_training_map.pkl', 'rb') as f:
#     data = pickle.load(f)

In [ ]:
print(hex(id(reaction_maps['Root'][0])))

In [ ]:
print(hex(id(reaction_maps['Root'][1])))

In [ ]:
reaction_maps['Root'][0].kinetics

In [ ]:
ensemble = np.load('Disproportionation_ensemble_matrix.npy')

cov = np.load('Disproportionation_covariance_matrix.npy')

with open('Disproportionation_training_map.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
ensemble.shape

In [ ]:
n_rules = len(database.kinetics.families['Disproportionation'].rules.entries)

i_train = 12
plt.hist(np.log(ensemble[n_rules + i_train, :]))

plt.axvline(x=np.log(data[i_train].get_rate_coefficient(1000)), color='black')

In [ ]:
len(data)

In [ ]:
len(database.kinetics.families['Disproportionation'].rules.entries)

In [ ]:
cov.shape

In [ ]:
plt.matshow(np.log(cov))
plt.colorbar()

In [ ]:
database.kinetics.families[family_name].depositories[0].match_node_to_structure()

In [ ]:
Tref = 1000.0
fmax = 1.0e5
recipe = database.kinetics.families[family_name].forward_recipe.actions

not_close = 0
# Get baseline for each node and make sure it's not too different from what's currently in the database
for node_name, rate_rule in database.kinetics.families[family_name].rules.entries.items():
    if not rate_rule[0].data.uncertainty.N == len(reaction_maps[node_name]):
        print(node_name)

    # fit the rule and compare to existing kinetics
    rxns = reaction_maps[node_name]    
    label = node_name
    ranks = [r.rank for r in reaction_maps[node_name]]
    rr = recipe, rxns, Tref, fmax, label, ranks

    new_kinetics = rmgpy.data.kinetics.family._make_rule(rr)
    if not new_kinetics.is_similar_to(rate_rule[0].data):
        # print(rate_rule)
        not_close += 1
print( not_close / len(database.kinetics.families[family_name].rules.entries))
        
        # Ts = np.linspace(300, 1500, 101)
        # new_ks = np.zeros_like(Ts)
        # old_ks = np.zeros_like(Ts)
        # delta_H = 0
        # for i in range(len(Ts)):
        #     new_ks[i] = new_kinetics.get_rate_coefficient(Ts[i], delta_H)
        #     old_ks[i] = old_kinetics.get_rate_coefficient(Ts[i], delta_H)
            
        
        # plt.plot(1.0 / Ts, np.log10(old_ks), label='old')
        # plt.plot(1.0 / Ts, np.log10(new_ks), label='new')
        # plt.show()



In [ ]:
rxns[0].kinetics

In [ ]:
# now test perturbing a node

node_name = example_node3

 # fit the rule and compare to existing kinetics
rxns = reaction_maps[node_name]    
label = node_name
ranks = [r.rank for r in reaction_maps[node_name]]
rr = recipe, rxns, Tref, fmax, label, ranks

old_kinetics = database.kinetics.families[family_name].rules.entries[node_name][0].data
new_kinetics = rmgpy.data.kinetics.family._make_rule(rr)


originalA = copy.deepcopy(rxns[0].kinetics.A)
rxns[0].kinetics.A.value_si *= 10
perturbed_kinetics = rmgpy.data.kinetics.family._make_rule(rr)
rxns[0].kinetics.A = originalA

Ts = np.linspace(300, 1500, 101)
new_ks = np.zeros_like(Ts)
old_ks = np.zeros_like(Ts)
p_ks = np.zeros_like(Ts)
delta_H = 0
for i in range(len(Ts)):
    new_ks[i] = new_kinetics.get_rate_coefficient(Ts[i], delta_H)
    old_ks[i] = old_kinetics.get_rate_coefficient(Ts[i], delta_H)
    p_ks[i] = perturbed_kinetics.get_rate_coefficient(Ts[i], delta_H)

plt.plot(1.0 / Ts, np.log10(old_ks), label='old')
plt.plot(1.0 / Ts, np.log10(new_ks), label='new')
plt.plot(1.0 / Ts, np.log10(p_ks), label='perturbed')
plt.legend()
plt.show()

In [ ]:
# try refitting a node
example_node1 = 'Root_N-4R->H_4CNOS-u1_1R!H->O_2R!H->C_Ext-4CNOS-R_N-Sp-5R!H=4CCNNOOSS_4CNOS->C_Sp-5R!H-4C_N-5R!H->C'
example_node3 = 'Root_N-4R->H_N-4CNOS-u1_1R!H->O'
len(reaction_maps[example_node3])

In [ ]:
old_kinetics = database.kinetics.families[family_name].rules.entries[example_node3][0].data

In [ ]:
new_kinetics = rmgpy.data.kinetics.family._make_rule(rr)

In [ ]:
Ts = np.linspace(300, 1500, 101)
new_ks = np.zeros_like(Ts)
old_ks = np.zeros_like(Ts)
delta_H = 0
for i in range(len(Ts)):
    new_ks[i] = new_kinetics.get_rate_coefficient(Ts[i], delta_H)
    old_ks[i] = old_kinetics.get_rate_coefficient(Ts[i], delta_H)
    

plt.plot(1.0 / Ts, np.log10(old_ks), label='old')
plt.plot(1.0 / Ts, np.log10(new_ks), label='new')